# Pré-processamento e Feature Engineering — Random Forest (KaIA)

Gera os conjuntos de treino/teste normalizados a partir da base sintética e salva os artefatos em `ml/artifacts/`. Rode com **Restart & Run All**.

## 1. Imports e carregamento da base

In [ ]:
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42

# Caminhos robustos: base bruta em <raiz>/data, saidas em <raiz>/ml/data.
# Funciona rodando de dentro de ml/ ou da raiz do projeto.
HERE = Path.cwd()
ROOT = HERE if (HERE / "data" / "KaIA_Base_Sintetica.xlsx").exists() else HERE.parent
DATA_SRC = ROOT / "data" / "KaIA_Base_Sintetica.xlsx"
OUT_DIR = ROOT / "ml" / "artifacts"
OUT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_excel(DATA_SRC)
print("Base carregada de:", DATA_SRC)
print("Saidas irao para: ", OUT_DIR)
df.head()

## 2. Inspeção: shape, tipos de coluna e valores nulos

In [ ]:
print("Shape:", df.shape)
print("\n--- Tipos de coluna ---")
print(df.dtypes)
print("\n--- Valores nulos por coluna ---")
print(df.isnull().sum())
print("\n--- Distribuicao do target ---")
print(df["target"].value_counts())

## 3. Limpeza: remoção de colunas e conversão de `horario_estudo`

Remove `session_id`, `perfil`, `tipo_atividade` e converte `horario_estudo` em `hora_do_dia` (0.0–23.99). Na base atual a coluna vem como *datetime*; mantemos um *fallback* para o caso de serial Excel.

In [ ]:
df = df.drop(columns=["session_id", "perfil", "tipo_atividade"])

col = df["horario_estudo"]
if pd.api.types.is_datetime64_any_dtype(col):            # caso atual (datetime)
    hora = col.dt.hour + col.dt.minute / 60 + col.dt.second / 3600
else:                                                     # fallback: serial Excel
    hora = (col.astype(float) % 1) * 24
df["hora_do_dia"] = hora.round(2)
df = df.drop(columns=["horario_estudo"])

print("Colunas apos limpeza:", list(df.columns))
df[["hora_do_dia"]].head()

## 4. Feature engineering: `produtividade` e `distracao_score`

`distracao_score` é uma média ponderada; `tempo_fora_foco_s` entra normalizado (min-max 0..1) para ficar na mesma escala dos demais termos.

In [ ]:
df["produtividade"] = df["acertos_questoes"] / df["duracao_sessao_min"]

tff = df["tempo_fora_foco_s"]
tff_norm = (tff - tff.min()) / (tff.max() - tff.min())
df["distracao_score"] = (
    0.4 * df["mudancas_aba"]
    + 0.3 * df["cliques_fora_area_estudo"]
    + 0.3 * tff_norm
)

df[["produtividade", "distracao_score"]].describe()

## 5. Encoding do target

`engajado=0`, `distraido=1`, `muito_distraido=2`.

In [ ]:
TARGET_MAP = {"engajado": 0, "distraido": 1, "muito_distraido": 2}
df["target"] = df["target"].map(TARGET_MAP)

assert df["target"].notna().all(), "Valor de target fora do mapa esperado!"
df["target"] = df["target"].astype(int)
print(df["target"].value_counts().sort_index())

## 6. Separar X e y

In [ ]:
y = df["target"]
X = df.drop(columns=["target"])

print("X:", X.shape, "| y:", y.shape)
print("Features (todas numericas):")
print(list(X.columns))

## 7. Split treino/teste (80/20, `random_state=42`)

Estratificado para preservar a proporção das 3 classes em treino e teste.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
print("Treino:", X_train.shape, "| Teste:", X_test.shape)

## 8. Normalização com `StandardScaler` (fit só no treino)

In [ ]:
scaler = StandardScaler()
X_train = pd.DataFrame(
    scaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index
)
X_test = pd.DataFrame(
    scaler.transform(X_test), columns=X_test.columns, index=X_test.index
)
print("Media ~0 e desvio ~1 no treino (amostra):")
print(X_train.mean().round(3).head(3))
print(X_train.std(ddof=0).round(3).head(3))

## 9. Salvar artefatos em `ml/artifacts/`

In [ ]:
artefatos = {
    "X_train.pkl": X_train,
    "X_test.pkl": X_test,
    "y_train.pkl": y_train,
    "y_test.pkl": y_test,
    "scaler.pkl": scaler,
}
for nome, obj in artefatos.items():
    caminho = OUT_DIR / nome
    with open(caminho, "wb") as f:
        pickle.dump(obj, f)
    print("salvo:", caminho)

## 10. Resumo final

In [ ]:
nomes = {0: "engajado", 1: "distraido", 2: "muito_distraido"}

print("=== SHAPES ===")
print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

print("\n=== Distribuicao do target - TREINO ===")
print(y_train.map(nomes).value_counts())

print("\n=== Distribuicao do target - TESTE ===")
print(y_test.map(nomes).value_counts())